# Commented notebook: `print_peetre_decomposition` and `apply()` with NUFFT, AAA & the unified joint representation

This notebook demonstrates the Peetre decomposition pipeline from `psiop.py`,
covering four joint-residual backends:

1. `joint_backend='direct'` — raw symbolic joint residual.
2. `joint_backend='nufft'` — NUFFT phase decomposition (O(N log N)),
   with automatic fallback to direct when the symbol is not oscillatory.
3. `joint_backend='lowrank'` — Chebyshev/SVD separable factorization.
4. `joint_backend='aaa'` — Vector-valued AAA rational approximation
   for resolvent/pole-shaped symbols (e.g. `1/(1+(x-ξ)²)`).
5. `joint_backend='auto'` — symbolic structure analysis routing to the best
   of the above.

New in this revision:
- `peetre_decomposition(classify_joint=True)` pre-classifies the joint
  residual (purely symbolic, no grid needed).
- A single representation layer `_resolve_joint_representation` shared by
  `apply_peetre`, `apply_hybrid` and `print_peetre_decomposition`.

It also shows how to numerically apply the operator via `apply()` /
`apply_peetre()` with each backend, and compares accuracy and timing.

## What `print_peetre_decomposition` does

The method calls `peetre_decomposition` and pretty-prints the result.
Joint-residual display (and execution) now go through the unified
representation resolver `_resolve_joint_representation`, which returns a
typed dict consumed identically by the printer and by `apply_peetre`.

| Option | Description |
|---|---|
| `joint_backend='direct'` | Print the raw joint residual terms. |
| `joint_backend='nufft'` | Resolve the NUFFT plan; print the detected phase structure (no bounds needed). |
| `joint_backend='lowrank'` | Factorize via Chebyshev/SVD and print the separable pairs. |
| `joint_backend='aaa'` | Build the AAA rational fit; print the fit quality. |
| `joint_backend='auto'` | Symbolically select the best backend, then print accordingly. |
| `joint_bounds` | Required only for `'lowrank'` and `'aaa'` printing (no grid available). |
| `joint_degree` | Chebyshev degree per variable (`'lowrank'`). |
| `joint_tol` | SVD truncation tolerance (`'lowrank'`) / AAA fit tolerance (`'aaa'`). |
| `separable_local` | Forwarded to `peetre_decomposition`. |

### Representation types

`_resolve_joint_representation(joint_symbol, backend, bounds, ...)` returns:

| `rep["type"]` | backend | content |
|---|---|---|
| `"separable_pairs"` | lowrank | `pairs=[(a_k, q_k)]`, `metrics` |
| `"nufft_plan"` | nufft | `plan_info` (grid-free; periodic-only at apply time) |
| `"aaa_callable"` | aaa | `symbol_func`, `metrics` |
| `"direct"` | direct | raw symbol |
| `"nufft_unrepresentable"` / `"aaa_unfit"` | — | signals fallback to direct |

### Fallback cascade (at apply time, in `_apply_joint_residual`)


NUFFT applies to joint symbols of the form
`c(x)·g(ξ)·exp(iλ(x)μ(ξ))`, e.g. `sin(x·ξ)`, `exp(i·x·ξ)`, chirps.
Rational or Gaussian joint kernels like `1/(1+(x−ξ)²)` are not
NUFFT-decomposable and resolve to `nufft_unrepresentable`.


## Imports

In [1]:
import time
import numpy as np

import sympy as sp
from psiop import PseudoDifferentialOperator

---
## 1D Symbolic Decomposition

We start with a 1D symbol containing all three Peetre classes:
- `xi**2` is **local** (polynomial in frequency).
- `x*sin(xi)` is **separable**: `a(x) * q(xi)`.
- `1/(1+(x-xi)**2)` is **genuinely joint** (not NUFFT-compatible).

In [2]:
x, xi = sp.symbols('x xi', real=True)

# 1D symbol with three Peetre classes.
# The joint term 1/(1+(x-xi)^2) is NOT NUFFT-decomposable.
p1 = xi**2 + x * sp.sin(xi) + 1 / (1 + (x - xi)**2)

op1 = PseudoDifferentialOperator(p1, [x], mode='symbol')

### Default printing: `joint_backend='direct'`

In [3]:
print('=' * 70)
print('DEFAULT: joint_backend=direct, separable_local=False')
print('=' * 70)
op1.print_peetre_decomposition()

DEFAULT: joint_backend=direct, separable_local=False
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


### Option: `separable_local=True`

Local polynomial terms are exposed as separable pairs `a(x)*q(xi)`.

In [4]:
print('separable_local=True')
op1.print_peetre_decomposition(separable_local=True)

separable_local=True
--- 0 local term(s), polynomial in (xi,) ---
--- 2 separable non-local term(s) ---
  (1) * (xi**2)
  (x) * (sin(xi))
--- 1 irreducible joint term(s) ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = 0
separable_symbol = x*sin(xi) + xi**2
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


### NEW: programmatic inspection — `classify_joint` and the unified representation

`peetre_decomposition(classify_joint=True)` runs the symbolic auto-selector at
decomposition time and stores the recommended backend in
`deco['joint_backend']` — no grid, no bounds.

`_resolve_joint_representation(...)` then normalizes the joint residual into
the typed executable form shared by `apply_peetre` and the printer. For `op1`
(joint part `1/(1+(x-ξ)²)`) we expect:

| requested backend | rep type |
|---|---|
| `'lowrank'` | `separable_pairs` |
| `'nufft'` | `nufft_unrepresentable` |
| `'aaa'` | `aaa_callable` |
| `'direct'` | `direct` |

In [5]:
# 1. Symbolic classification stored in the decomposition (no grid needed):
deco = op1.peetre_decomposition(classify_joint=True)
print('recommended joint backend:', deco.get('joint_backend'))   # -> 'aaa'

# 2. Normalize the joint residual into the unified representation:
bounds = {x: (-5, 5), xi: (-30, 30)}
for backend in ('lowrank', 'nufft', 'aaa', 'direct'):
    rep = op1._resolve_joint_representation(
        deco['joint_symbol'], backend=backend, bounds=bounds,
    )
    line = f"backend={backend:8s} -> type={rep['type']}"
    if rep['type'] == 'separable_pairs':
        line += (f"  (pairs={len(rep['pairs'])}, "
                 f"rel_l2_error={rep['metrics']['rel_l2_error']:.3e})")
    elif rep['type'] == 'aaa_callable':
        line += f"  (rel_l2_error={rep['metrics']['rel_l2_error']:.3e})"
    print(line)

recommended joint backend: aaa
backend=lowrank  -> type=separable_pairs  (pairs=5, rel_l2_error=1.432e+00)
backend=nufft    -> type=nufft_unrepresentable
backend=aaa      -> type=aaa_unfit
backend=direct   -> type=direct


### Option: `joint_backend='lowrank'`

The joint residual is approximated on a bounded rectangle by
`p_joint(x, xi) ≈ Σ_k a_k(x) q_k(xi)`.
`joint_bounds` is required because no numerical grid is available here.

In [6]:
print('joint_backend=lowrank')
op1.print_peetre_decomposition(
    joint_backend='lowrank',
    joint_bounds={x: (-5, 5), xi: (-30, 30)},
    joint_degree=6,
)

joint_backend=lowrank
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=1.432e+00) ---
  (0.00026926*x**6 - 0.013732*x**4 + 0.21358*x**2 - 1.0019) * (7.2629e-9*xi**6 - 1.3078e-5*xi**4 + 0.0069955*xi**2 - 0.99805)
  (1.0742e-5*x**6 - 0.00054136*x**4 + 0.0091446*x**2 + 0.011449) * (9.7625e-10*xi**6 - 1.5949e-6*xi**4 + 0.00065437*xi**2 + 0.0093696)
  (6.8068e-7*x**5 + 7.4843e-5*x**3 + 0.0091005*x) * (1.1576e-8*xi**5 - 1.9081e-5*xi**3 + 0.0080346*xi)
  (5.5743e-6*x**6 - 0.00029603*x**4 + 0.0036664*x**2 - 0.0026623) * (-1.6396e-10*xi**6 + 2.3675e-7*xi**4 - 7.4572e-5*xi**2 + 0.0047266)
  (-1.3656e-6*x**5 - 0.00010283*x**3 + 0.0025124*x) * (-1.3111e-9*xi**5 + 1.7515e-6*xi**3 - 0.00039665*xi)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


### `joint_backend='nufft'` — oscillatory joint symbols

The NUFFT backend targets joint residuals of the form

```
p_joint(x, ξ) = Σ_k c_k(x) · g_k(ξ) · exp(i λ_k(x) μ_k(ξ))
```

Typical examples: `sin(x·ξ)`, `exp(i·x·ξ)`, chirps `exp(i·x²·ξ)`.

This symbol is NUFFT-friendly: the joint part `sin(x*xi)` decomposes
into two exponential terms via Euler's formula.

In [7]:
x, xi = sp.symbols('x xi', real=True)

# Joint part sin(x xi) IS NUFFT-decomposable:
#   sin(x xi) = (exp(i x xi) - exp(-i x xi)) / (2i)
# Each exponential is c(x)*g(xi) exp(i lambda(x)*mu(xi))
# with lambda(x)=x, mu(xi)=xi.
p_nufft = xi**2 + x * sp.sin(xi) + sp.sin(x * xi)

op_nufft = PseudoDifferentialOperator(p_nufft, [x], mode='symbol')

print('=' * 70)
print('NUFFT-friendly symbol: joint part = sin(x*xi)')
print('=' * 70)
op_nufft.print_peetre_decomposition(joint_backend='nufft')

NUFFT-friendly symbol: joint part = sin(x*xi)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- NUFFT structure detected (1d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = sin(x*xi)


### NUFFT fallback: non-oscillatory joint symbol

The rational kernel `1/(1+(x−ξ)²)` is not of the form
`c(x)·g(ξ)·exp(iλ(x)μ(ξ))`, so `_resolve_nufft_plan` returns `None` and the
representation is `nufft_unrepresentable`: the printer shows the raw joint
terms, and at apply time `_apply_joint_residual` warns and falls back to the
exact direct KN quadrature.

In [8]:
# This joint part is NOT NUFFT-decomposable: the representation resolves
# to 'nufft_unrepresentable' -> printing shows raw terms; apply() warns
# and falls back to direct.
print('=' * 70)
print('NUFFT fallback: joint part = 1/(1+(x-xi)^2)')
print('=' * 70)
op1.print_peetre_decomposition(joint_backend='nufft')

NUFFT fallback: joint part = 1/(1+(x-xi)^2)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual: backend 'nufft' could not represent the symbol. Raw joint term(s): ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


### NUFFT with a chirp and separable amplitude

`x**2 * exp(i*x*xi) * cos(xi)` has:
- spatial amplitude `c(x) = x**2`,
- spectral factor `g(ξ) = cos(ξ)`,
- phase `λ(x)·μ(ξ) = x·ξ`.

This is a textbook NUFFT-compatible term.

In [9]:
x, xi = sp.symbols('x xi', real=True)

p_chirp = xi**2 + x**2 * sp.exp(sp.I * x * xi) * sp.cos(xi)
op_chirp = PseudoDifferentialOperator(p_chirp, [x], mode='symbol')

print('Chirp symbol: joint part = x^2 * exp(i x xi) * cos(xi)')
op_chirp.print_peetre_decomposition(joint_backend='nufft')

Chirp symbol: joint part = x^2 * exp(i x xi) * cos(xi)
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 0 separable non-local term(s) ---
--- NUFFT structure detected (1d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = xi**2
separable_symbol = 0
joint_symbol = x**2*exp(I*x*xi)*cos(xi)


### Advanced NUFFT: Non-linear chirps and multi-phase sums

The NUFFT backend isn't limited to simple $\sin(x\xi)$ terms. It handles:
1. **Non-linear chirps**: Phases like $\exp(i \Lambda(x) \mu(\xi))$ where $\Lambda(x)$ is a polynomial (e.g., $x^2+x$).
2. **Amplitude modulation**: Spatial and spectral envelopes $c(x)g(\xi)$ wrapping the phase.
3. **Multi-term sums**: Linear combinations of different oscillatory phases.

In [10]:
x, xi = sp.symbols('x xi', real=True)

# 1. Non-linear chirp with Gaussian spectral envelope and cubic spatial amplitude
# Phase: Lambda(x) = x^2 + x, M(xi) = xi
# c(x) = x^3, g(xi) = exp(-xi^2)
chirp_term = x**3 * sp.exp(-xi**2) * sp.exp(sp.I * (x**2 + x) * xi)

# 2. A standard oscillatory term (cos(x xi) = 0.5 exp(ix xi) + 0.5 exp(-ix xi))
osc_term = sp.cos(x * xi)

p_nufft_adv = chirp_term + osc_term
op_nufft_adv = PseudoDifferentialOperator(p_nufft_adv, [x], mode='symbol')

print('=' * 70)
print('Advanced 1D NUFFT: Chirp + Oscillatory sum')
print('=' * 70)
op_nufft_adv.print_peetre_decomposition(joint_backend='nufft')

Advanced 1D NUFFT: Chirp + Oscillatory sum
--- 0 local term(s), polynomial in (xi,) ---
--- 0 separable non-local term(s) ---
--- NUFFT structure detected (1d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = 0
separable_symbol = 0
joint_symbol = x**3*exp(-xi**2)*exp(I*x*xi)*exp(I*x**2*xi) + cos(x*xi)


### Smoother joint kernel in 1D (low-rank comparison)

A Gaussian bump `exp(−(x−ξ)²/8)` is smooth and well suited to
low-rank Chebyshev/SVD approximation, but not NUFFT-decomposable.

In [11]:
x, xi = sp.symbols('x xi', real=True)

p1b = xi**2 + x * sp.sin(xi) + sp.exp(-((x - xi)**2) / 8)
op1b = PseudoDifferentialOperator(p1b, [x], mode='symbol')

### Comparing bounds and degrees for the low-rank approximation

In [12]:
for bounds, deg in [
    ({x: (-5, 5), xi: (-15, 15)}, 8),
    ({x: (-4, 4), xi: (-12, 12)}, 10),
]:
    print('bounds =', bounds, ', degree =', deg)
    op1b.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds=bounds,
        joint_degree=deg,
    )
    print()

bounds = {x: (-5, 5), xi: (-15, 15)} , degree = 8
--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual factorized into 5 low-rank term(s) via factorize_symbolic (rel_l2_error=4.352e-01) ---
  (-3.4432e-7*x**7 + 0.00017519*x**5 - 0.0078706*x**3 - 0.026926*x) * (5.4597e-8*xi**7 - 2.9021e-5*xi**5 + 0.0049665*xi**3 - 0.27015*xi)
  (-2.0444e-6*x**8 + 0.00017141*x**6 - 0.0059783*x**4 + 0.11081*x**2 - 0.8798) * (-8.2942e-9*xi**8 + 4.6243e-6*xi**6 - 0.00087012*xi**4 + 0.0611*xi**2 - 1.1139)
  (9.7261e-7*x**8 - 8.6251e-5*x**6 + 0.0021714*x**4 + 0.0032102*x**2 + 0.15204) * (-8.3651e-9*xi**8 + 4.4135e-6*xi**6 - 0.00074419*xi**4 + 0.038716*xi**2 + 0.12941)
  (2.266e-6*x**7 + 1.4903e-5*x**5 - 0.0011314*x**3 - 0.0048597*x) * (8.219e-9*xi**7 - 3.7058e-6*xi**5 + 0.00047073*xi**3 - 0.011925*xi)
  (-1.6054e-7*x**8 - 9.6843e-6*x**6 + 6.9195e-5*x**4 + 0.0054866*x**2 - 0.015388) * (-9.3695e-10*xi**8 + 4.3579e-7*xi**6 -

---
## 1D Numerical Application & AAA Backend

We now apply the operator numerically to a test function using
`apply()` (which dispatches to `apply_peetre()` when `backend='peetre'`).

The joint backends are compared:

| Backend | Strategy | Cost |
|---|---|---|
| `'direct'` | Full KN quadrature on the joint symbol | O(N²) |
| `'nufft'` | NUFFT phase decomposition (if applicable) | O(N log N) |
| `'lowrank'` | Chebyshev/SVD separable pairs | O(r · N log N) |
| `'aaa'` | Vector-valued AAA rational fit | O(N log N) via fast callable |

We use `joint_backend='direct'` as the reference and measure the
error of the other backends against it.

In [13]:
# --- Grid setup ---
N = 256
L = 5.0
x_grid = np.linspace(-L, L, N, endpoint=False)
dx = x_grid[1] - x_grid[0]
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

# --- Test function: a Gaussian bump ---
u = np.exp(-x_grid**2 / 2).astype(complex)

# --- NUFFT-friendly symbol: joint part = sin(x xi) ---
x, xi = sp.symbols('x xi', real=True)
p_apply = xi**2 + x * sp.sin(xi) + sp.sin(x * xi)
op_apply = PseudoDifferentialOperator(p_apply, [x], mode='symbol')

print(f'Grid: N={N}, L={L}, dx={dx:.4f}')
print(f'Symbol: {p_apply}')

Grid: N=256, L=5.0, dx=0.0391
Symbol: x*sin(xi) + xi**2 + sin(x*xi)


In [14]:
# --- Reference: direct joint application ---
t0 = time.time()
res_direct = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
t_direct = time.time() - t0

# --- NUFFT backend ---
t0 = time.time()
res_nufft = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='nufft',
    freq_window=None, clamp=np.inf,
)
t_nufft = time.time() - t0

# --- Low-rank backend (bounds inferred from grid) ---
t0 = time.time()
res_lowrank = op_apply.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='lowrank',
    joint_degree=8,
    freq_window=None, clamp=np.inf,
)
t_lowrank = time.time() - t0

# --- Error vs. direct reference ---
err_nufft = np.max(np.abs(res_nufft - res_direct))
err_lowrank = np.max(np.abs(res_lowrank - res_direct))

print(f'direct   : time = {t_direct:.4f}s')
print(f'nufft    : time = {t_nufft:.4f}s,  max |err| vs direct = {err_nufft:.3e}')
print(f'lowrank  : time = {t_lowrank:.4f}s,  max |err| vs direct = {err_lowrank:.3e}')

direct   : time = 0.0250s
nufft    : time = 0.2385s,  max |err| vs direct = 6.877e-13
lowrank  : time = 0.4694s,  max |err| vs direct = 5.153e-01


### `apply()` with a non-NUFFT joint symbol

The rational kernel `1/(1+(x−ξ)²)` cannot be decomposed by NUFFT.
When `joint_backend='nufft'` is requested, `_apply_joint_residual` receives a
`nufft_unrepresentable` representation, emits a warning and falls back to the
exact direct joint application.

In [15]:
# Joint part is NOT NUFFT-decomposable -> cascade: nufft -> lowrank -> direct
p_fallback = xi**2 + x * sp.sin(xi) + 1 / (1 + (x - xi)**2)
op_fallback = PseudoDifferentialOperator(p_fallback, [x], mode='symbol')

print('Requesting nufft backend on a non-NUFFT symbol (expect warnings):')
res_fb = op_fallback.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='nufft',
    joint_degree=6,
    freq_window=None, clamp=np.inf,
)

# Compare against pure direct
res_fb_direct = op_fallback.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
print(f'max |fallback - direct| = {np.max(np.abs(res_fb - res_fb_direct)):.3e}')

Requesting nufft backend on a non-NUFFT symbol (expect warnings):
max |fallback - direct| = 0.000e+00


/home/philippe/psipy/src/psiop.py:2594: UserWarning: Joint residual does not classify as NUFFT-representable (no oscillatory phase of the form exp(i*Lambda(x)*M(xi)) found). Falling back to direct joint application.
  warnings.warn(


### `joint_backend='aaa'` — rational and resolvent-shaped symbols

The **AAA** (Adaptive Antoulas-Algoet) backend targets joint residuals that are 
**rational or have algebraic decay/poles** (e.g., `1/(1+(x-ξ)²)`), which are 
neither separable nor NUFFT-compatible (no oscillatory phase). 

It builds a compact rational approximation of the symbol via vector-valued AAA, 
shared across a Chebyshev grid in the spatial variable. This is highly efficient 
for symbols with fixed or slowly varying pole locations.

In [16]:
# Rational joint symbol: 1 / (1 + (x - xi)**2)
p_aaa = xi**2 + x * sp.sin(xi) + 1 / (1 + (x - xi)**2)
op_aaa = PseudoDifferentialOperator(p_aaa, [x], mode='symbol')

print('=' * 70)
print('AAA-friendly symbol: joint part = 1/(1+(x-xi)^2)')
print('=' * 70)

# --- AAA backend application ---
t0 = time.time()
res_aaa = op_aaa.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='aaa',
    joint_tol=1e-6,
    freq_window=None, clamp=np.inf,
)
t_aaa = time.time() - t0

# --- Direct reference for op_aaa ---
res_aaa_direct = op_aaa.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)

err_aaa = np.max(np.abs(res_aaa - res_aaa_direct))
print(f'aaa      : time = {t_aaa:.4f}s,  max |err| vs direct = {err_aaa:.3e}')

AAA-friendly symbol: joint part = 1/(1+(x-xi)^2)
aaa      : time = 3.0889s,  max |err| vs direct = 0.000e+00


/home/philippe/psipy/src/psiop.py:2636: UserWarning: Joint residual could not be fit by AAA to the requested tolerance (joint_tol). This can happen for symbols whose poles move with x/y (a genuinely different, diagonal-singularity structural class). Falling back to direct joint application.
  warnings.warn(


### Printing the AAA representation

`print_peetre_decomposition(joint_backend='aaa', joint_bounds=...)` resolves
the AAA callable through the same representation layer and reports the fit
quality (bounds are required here, as for `'lowrank'`, since no grid is
available).

In [17]:
op_aaa.print_peetre_decomposition(
    joint_backend='aaa',
    joint_bounds={x: (-5, 5), xi: (-30, 30)},
    joint_tol=1e-6,
)

--- 1 local term(s), represented as a(x)*q(xi) ---
  (1) * (xi**2)
--- 1 separable non-local term(s) ---
  (x) * (sin(xi))
--- joint residual: backend 'aaa' could not represent the symbol. Raw joint term(s): ---
  1/(x**2 - 2*x*xi + xi**2 + 1)
local_symbol = xi**2
separable_symbol = x*sin(xi)
joint_symbol = 1/(x**2 - 2*x*xi + xi**2 + 1)


### Advanced AAA: Fixed poles, Multi-poles, and the "Moving Pole" Trap

The AAA backend excels at rational functions where the **poles in the frequency domain are fixed** (independent of $x$). 
It builds a shared pole structure across a Chebyshev grid in $x$.

**The Trap (Diagonal Singularities):** 
If the poles *move* with the spatial variable (e.g., $1/(\xi - x - i\epsilon)$), the shared-pole AAA approximation requires an excessive number of poles to converge. The backend's quality gate will detect this and automatically fall back to the exact (but slower) `direct` method.

In [18]:
x, xi = sp.symbols('x xi', real=True)

# 1. Fixed-pole multi-pole rational (AAA will SUCCEED and compress)
p_aaa_fixed = (x**2 + 1) / (xi**4 + xi**2 + 1) + x / (xi**2 + 4)
op_aaa_fixed = PseudoDifferentialOperator(p_aaa_fixed, [x], mode='symbol')

# 2. Moving-pole rational / Diagonal singularity (AAA will FAIL and FALLBACK)
p_aaa_moving = 1 / (xi - x - sp.I)
op_aaa_moving = PseudoDifferentialOperator(p_aaa_moving, [x], mode='symbol')

print('=' * 70)
print('AAA Test 1: Fixed-pole rational (Expected: Success)')
print('=' * 70)
t0 = time.time()
res_aaa_fix = op_aaa_fixed.apply(
    u, x_grid, kx, boundary_condition='periodic',
    joint_backend='aaa', joint_tol=1e-6, freq_window=None, clamp=np.inf,
)
t_aaa_fix = time.time() - t0
res_dir_fix = op_aaa_fixed.apply(u, x_grid, kx, boundary_condition='periodic', joint_backend='direct')
print(f'AAA fixed : time = {t_aaa_fix:.4f}s, max |err| = {np.max(np.abs(res_aaa_fix - res_dir_fix)):.3e}')

print('\n' + '=' * 70)
print('AAA Test 2: Moving-pole / Diagonal singularity (Expected: Fallback warning)')
print('=' * 70)
t0 = time.time()
res_aaa_mov = op_aaa_moving.apply(
    u, x_grid, kx, boundary_condition='periodic',
    joint_backend='aaa', joint_tol=1e-6, freq_window=None, clamp=np.inf,
)
t_aaa_mov = time.time() - t0
res_dir_mov = op_aaa_moving.apply(u, x_grid, kx, boundary_condition='periodic', joint_backend='direct')
print(f'AAA moving: time = {t_aaa_mov:.4f}s, max |err| = {np.max(np.abs(res_aaa_mov - res_dir_mov)):.3e}')

AAA Test 1: Fixed-pole rational (Expected: Success)
AAA fixed : time = 0.0548s, max |err| = 5.138e-08

AAA Test 2: Moving-pole / Diagonal singularity (Expected: Fallback warning)
AAA moving: time = 2.3599s, max |err| = 8.014e-08


---
## 2D Symbolic Decomposition

The symbol contains:
- local: `xi**2 + eta**2`,
- separable: `x*y*cos(xi + eta)`,
- joint Gaussian: `exp(−((x−ξ)² + (y−η)²)/8)`.

In [19]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

p2 = (
    xi**2
    + eta**2
    + x * y * sp.cos(xi + eta)
    + sp.exp(-((x - xi)**2 + (y - eta)**2) / 8)
)

op2 = PseudoDifferentialOperator(p2, [x, y], mode='symbol')

### Default 2D printing

In [20]:
print('DEFAULT 2D: joint_backend=direct')
op2.print_peetre_decomposition()

DEFAULT 2D: joint_backend=direct
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- 1 irreducible joint term(s) ---
  exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)
local_symbol = eta**2 + xi**2
separable_symbol = x*y*cos(eta + xi)
joint_symbol = exp(-eta**2/8)*exp(-x**2/8)*exp(-xi**2/8)*exp(-y**2/8)*exp(eta*y/4)*exp(x*xi/4)


### 2D NUFFT: `sin((x+y)·ξ)` — genuinely joint phase

`sin((x+y)*xi)` has `Λ(x,y) = x+y` and `M(ξ,η) = ξ`, which embeds
into 3D (within finufft's `nufft3d3` capability) → NUFFT passes.

By contrast, `exp(i·x·ξ) + exp(i·y·η)` has two independent 
axis couplings that would need a 4D embedding → NUFFT rejects it
and falls back.

In [21]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# NUFFT-friendly 2D joint part: Lambda(x,y) = x+y, M(xi,eta) = xi
p2_nufft = (
    xi**2 + eta**2
    + x * y * sp.cos(xi + eta)
    + sp.sin((x + y) * xi)
)
op2_nufft = PseudoDifferentialOperator(p2_nufft, [x, y], mode='symbol')

print('2D NUFFT-friendly: joint part = sin((x+y) xi)')
op2_nufft.print_peetre_decomposition(joint_backend='nufft')

2D NUFFT-friendly: joint part = sin((x+y) xi)
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- NUFFT structure detected (2d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = eta**2 + xi**2
separable_symbol = x*y*cos(eta + xi)
joint_symbol = sin(x*xi + xi*y)


In [22]:
# Two independent axis couplings -> needs 4D -> NUFFT rejects, falls back
p2_reject = (
    xi**2 + eta**2
    + sp.exp(sp.I * x * xi) + sp.exp(sp.I * y * eta)
)
op2_reject = PseudoDifferentialOperator(p2_reject, [x, y], mode='symbol')

print('2D NUFFT rejection: joint = exp(i x xi) + exp(i y eta)')
print('(Expect fallback warning)')
op2_reject.print_peetre_decomposition(joint_backend='nufft')

2D NUFFT rejection: joint = exp(i x xi) + exp(i y eta)
(Expect fallback warning)
--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 0 separable non-local term(s) ---
--- NUFFT structure detected (2d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = eta**2 + xi**2
separable_symbol = 0
joint_symbol = exp(I*eta*y) + exp(I*x*xi)


### 2D low-rank joint printing and timing

In [23]:
for deg in [2, 3, 4]:
    t0 = time.time()
    op2.print_peetre_decomposition(
        joint_backend='lowrank',
        joint_bounds={x: (-3, 3), y: (-3, 3), xi: (-3, 3), eta: (-3, 3)},
        joint_degree=deg,
        joint_num_samples=3000,
    )
    print('degree', deg, 'time', time.time() - t0)
    print()

--- 1 local term(s), represented as a(x)*q(xi, eta) ---
  (1) * (eta**2 + xi**2)
--- 1 separable non-local term(s) ---
  (x*y) * (cos(eta + xi))
--- joint residual factorized into 9 low-rank term(s) via factorize_symbolic (rel_l2_error=2.859e-01) ---
  (-0.0029295*x**2*y**2 + 0.052254*x**2 + 0.052254*y**2 - 0.93207) * (-0.0029295*eta**2*xi**2 + 0.052254*eta**2 + 0.052254*xi**2 - 0.93207)
  (-0.011264*x**2*y + 0.0058369*x*y**2 - 0.10411*x + 0.20091*y) * (0.0058369*eta**2*xi - 0.011264*eta*xi**2 + 0.20091*eta - 0.10411*xi)
  (0.0058369*x**2*y + 0.011264*x*y**2 - 0.20091*x - 0.10411*y) * (0.011264*eta**2*xi + 0.0058369*eta*xi**2 - 0.10411*eta - 0.20091*xi)
  (-0.054938*x*y) * (-0.054938*eta*xi)
  (-4.9427e-5*x**2*y**2 - 0.049184*x**2 + 0.050213*y**2 - 0.0026288) * (-4.9427e-5*eta**2*xi**2 + 0.050213*eta**2 - 0.049184*xi**2 - 0.0026288)
  (0.006691*x**2*y**2 - 0.070016*x**2 - 0.069282*y**2 + 0.35586) * (0.006691*eta**2*xi**2 - 0.069282*eta**2 - 0.070016*xi**2 + 0.35586)
  (0.0085603*x**2*y

### 2D AAA: Rational symbols in 2D phase space

In 2D, the AAA backend uses a sequential vector-valued approach: it first compresses the $\xi$ dependence via shared poles, then compresses the $\eta$ dependence from the exact symbolic slices at the $\xi$ support points.

In [24]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# 2D Rational symbol: Poles lie on the unit circle in (xi, eta) space, 
# independent of (x, y). Amplitude is a simple polynomial x y.
p_2d_aaa = (x * y) / (xi**2 + eta**2 + 1)
op_2d_aaa = PseudoDifferentialOperator(p_2d_aaa, [x, y], mode='symbol')

print('=' * 70)
print('2D AAA: Rational symbol (x y)/(xi^2 + eta^2 + 1)')
print('=' * 70)

# Setup 2D grids for application
N2 = 256; L2 = 3.0
x_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
y_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
dx2 = x_grid2[1] - x_grid2[0]; dy2 = y_grid2[1] - y_grid2[0]
kx2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dy2)
X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2 = np.exp(-(X2**2 + Y2**2) / 2).astype(complex)

t0 = time.time()
res_2d_aaa = op_2d_aaa.apply(
    u2, x_grid2, kx2, y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='aaa', joint_tol=1e-5,
    freq_window=None, clamp=np.inf,
)
t_2d_aaa = time.time() - t0

res_2d_dir = op_2d_aaa.apply(
    u2, x_grid2, kx2, y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic', joint_backend='direct'
)

err_2d_aaa = np.max(np.abs(res_2d_aaa - res_2d_dir))
print(f'2D AAA    : time = {t_2d_aaa:.4f}s, max |err| vs direct = {err_2d_aaa:.3e}')

2D AAA: Rational symbol (x y)/(xi^2 + eta^2 + 1)
2D AAA    : time = 0.0267s, max |err| vs direct = 4.048e-08


---
## 2D Numerical Application

We apply the 2D operator to a Gaussian test field and compare the
three joint backends. For the NUFFT-friendly variant we use
`sin((x+y)·ξ)` as the joint part.

In [25]:
# --- 2D grid setup ---
N2 = 32
L2 = 3.0
x_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
y_grid2 = np.linspace(-L2, L2, N2, endpoint=False)
dx2 = x_grid2[1] - x_grid2[0]
dy2 = y_grid2[1] - y_grid2[0]
kx2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dx2)
ky2 = 2.0 * np.pi * np.fft.fftfreq(N2, d=dy2)

# --- Test function: 2D Gaussian ---
X2, Y2 = np.meshgrid(x_grid2, y_grid2, indexing='ij')
u2 = np.exp(-(X2**2 + Y2**2) / 2).astype(complex)

# --- NUFFT-friendly 2D symbol ---
x, y, xi, eta = sp.symbols('x y xi eta', real=True)
p2_apply = xi**2 + eta**2 + x * y * sp.cos(xi + eta) + sp.sin((x + y) * xi)
op2_apply = PseudoDifferentialOperator(p2_apply, [x, y], mode='symbol')

print(f'2D grid: N={N2}, L={L2}')

2D grid: N=32, L=3.0


In [26]:
# --- Reference: direct ---
t0 = time.time()
res2_direct = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
t2_direct = time.time() - t0

# --- NUFFT ---
t0 = time.time()
res2_nufft = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='nufft',
    freq_window=None, clamp=np.inf,
)
t2_nufft = time.time() - t0

# --- Low-rank ---
t0 = time.time()
res2_lowrank = op2_apply.apply(
    u2, x_grid2, kx2,
    y_grid=y_grid2, ky=ky2,
    boundary_condition='periodic',
    joint_backend='lowrank',
)
t2_lowrank = time.time() - t0

# --- Errors ---
err2_nufft = np.max(np.abs(res2_nufft - res2_direct))
err2_lowrank = np.max(np.abs(res2_lowrank - res2_direct))

print(f'direct   : time = {t2_direct:.4f}s')
print(f'nufft    : time = {t2_nufft:.4f}s,  max |err| = {err2_nufft:.3e}')
print(f'lowrank  : time = {t2_lowrank:.4f}s,  max |err| = {err2_lowrank:.3e}')

direct   : time = 0.1306s
nufft    : time = 0.3299s,  max |err| = 6.275e-13
lowrank  : time = 2.3145s,  max |err| = 5.358e-01


---
## Advanced Features

### Bonus: `apply()` with Weyl quantization

When the operator is built with `quantization='weyl'`, `apply_peetre()` 
automatically converts the Weyl symbol to its Kohn–Nirenberg equivalent
via `weyl_to_kn_symbol()` before decomposing.

Example: the Weyl symbol `x*xi` becomes the KN symbol `x*xi − i/2`.

In [27]:
x, xi = sp.symbols('x xi', real=True)

# Weyl-quantized operator
p_weyl = x * xi + sp.sin(xi)
op_weyl = PseudoDifferentialOperator(
    p_weyl, [x], mode='symbol', quantization='weyl'
)

# Show the KN correction
kn_symbol = op_weyl.weyl_to_kn_symbol(order=4)
print(f'Weyl symbol:  {p_weyl}')
print(f'KN equivalent: {kn_symbol}')

# Apply with Peetre backend (Weyl -> KN conversion is automatic)
res_weyl = op_weyl.apply(
    u, x_grid, kx,
    boundary_condition='periodic',
    joint_backend='direct',
    freq_window=None, clamp=np.inf,
)
print(f'apply() result shape: {res_weyl.shape}')
print(f'apply() result norm:  {np.linalg.norm(res_weyl):.6e}')

Weyl symbol:  x*xi + sin(xi)
KN equivalent: x*xi + sin(xi) - I/2
apply() result shape: (256,)
apply() result norm:  6.085112e+00


### Intelligent Backend Selection: `joint_backend='auto'`

Manually selecting the optimal joint backend (`'nufft'`, `'aaa'`, or `'lowrank'`) requires analyzing the mathematical structure of the joint residual. To simplify this, the `apply()` and `print_peetre_decomposition()` methods now support `joint_backend='auto'`.

When `'auto'` is requested, the engine programmatically inspects the symbolic joint residual and dispatches to the most efficient algorithm:

| Symbolic Structure | Auto-Selected Backend | Why? |
|---|---|---|
| **Oscillatory phase** (e.g., $\sin(x\xi)$, $e^{ix\xi}$) | `'nufft'` | Extracts the phase $\Lambda(x)M(\xi)$ for $O(N \log N)$ NUFFT transforms. |
| **Rational / Poles** (e.g., $1/(1+(x-\xi)^2)$) | `'aaa'` | Detects denominators/negative powers and builds a shared-pole AAA rational fit. |
| **Smooth non-oscillatory** (e.g., Gaussians) | `'lowrank'` | Falls back to Chebyshev/SVD separable factorization for smooth kernels. |

This ensures you always get the $O(N \log N)$ or $O(r N \log N)$ fast path when the math allows it, while safely falling back to exact quadrature if the quality gates fail.

In [28]:
def demo_auto_backend(name, p_expr):
    """
    Demonstrates the 'auto' backend selection by inspecting the symbol,
    revealing the chosen backend, and benchmarking it against 'direct'.
    """
    op = PseudoDifferentialOperator(p_expr, [x], mode='symbol')

    print(f"\n{'=' * 70}")
    print(f"CASE: {name}")
    print(f"Symbol: {p_expr}")
    print('=' * 70)

    # 1. Symbolic Inspection
    print("\n[1] Symbolic Peetre Decomposition (joint_backend='auto'):")
    print('-' * 50)
    op.print_peetre_decomposition(joint_backend='auto')

    # 2. Reveal the Auto-Selector's internal choice, now stored in the
    #    decomposition itself (no private helper call needed):
    deco = op.peetre_decomposition(classify_joint=True)
    joint_sym = deco['joint_symbol']
    if not op._peetre_is_zero(joint_sym):
        print(f"\n--> [Auto-Selector] Internally routed to backend: "
              f"'{deco.get('joint_backend')}'")
    else:
        print("\n--> [Auto-Selector] No joint residual found.")

    # 3. Numerical Application & Benchmark
    print("\n[2] Numerical Benchmark:")
    print('-' * 50)

    t0 = time.time()
    res_auto = op.apply(u, x_grid, kx, boundary_condition='periodic',
                        joint_backend='auto', freq_window=None, clamp=np.inf)
    t_auto = time.time() - t0

    t0 = time.time()
    res_dir = op.apply(u, x_grid, kx, boundary_condition='periodic',
                       joint_backend='direct', freq_window=None, clamp=np.inf)
    t_dir = time.time() - t0

    err = np.max(np.abs(res_auto - res_dir))
    print(f"  • 'auto'   execution time: {t_auto:.4f}s")
    print(f"  • 'direct' execution time: {t_dir:.4f}s")
    print(f"  • Max |err| vs direct   : {err:.3e}")


# Run the demonstrations for the 3 structural classes
x, xi = sp.symbols('x xi', real=True)

# 1. Oscillatory -> Auto-selects 'nufft'
demo_auto_backend("Oscillatory Phase (NUFFT)", sp.sin(x * xi))
# 2. Rational/Pole -> Auto-selects 'aaa'
demo_auto_backend("Rational / Resolvent (AAA)", 1 / (1 + (x - xi)**2))
# 3. Smooth Gaussian -> Auto-selects 'lowrank'
demo_auto_backend("Smooth Gaussian (Low-Rank)", sp.exp(-((x - xi)**2) / 8))


CASE: Oscillatory Phase (NUFFT)
Symbol: sin(x*xi)

[1] Symbolic Peetre Decomposition (joint_backend='auto'):
--------------------------------------------------
--- 0 local term(s), polynomial in (xi,) ---
--- 0 separable non-local term(s) ---
--- NUFFT structure detected (1d): oscillatory phase exp(i*Lambda(x)*M(xi)). No separable pairs to print (use apply() to execute). ---
local_symbol = 0
separable_symbol = 0
joint_symbol = sin(x*xi)

--> [Auto-Selector] Internally routed to backend: 'nufft'

[2] Numerical Benchmark:
--------------------------------------------------
  • 'auto'   execution time: 0.2475s
  • 'direct' execution time: 0.0082s
  • Max |err| vs direct   : 6.876e-13

CASE: Rational / Resolvent (AAA)
Symbol: 1/((x - xi)**2 + 1)

[1] Symbolic Peetre Decomposition (joint_backend='auto'):
--------------------------------------------------
--- 0 local term(s), polynomial in (xi,) ---
--- 0 separable non-local term(s) ---
--- detected 'aaa' structure; joint_bounds required to 

/home/philippe/psipy/src/psiop.py:2636: UserWarning: Joint residual could not be fit by AAA to the requested tolerance (joint_tol). This can happen for symbols whose poles move with x/y (a genuinely different, diagonal-singularity structural class). Falling back to direct joint application.
  warnings.warn(


## Hybrid Auto-Routing: Mixed Symbols

If a symbol contains a mix of NUFFT, AAA, and Lowrank structures, a monolithic `joint_backend='auto'` will fail to find a global pattern and fallback to the slow $O(N^2)$ `'direct'` method. 

By using `apply_hybrid()`, the engine splits the joint residual into its additive terms and routes each one to its mathematically optimal backend, preserving the $O(N \log N)$ speedup across the board.

In [29]:
import time
import numpy as np
import sympy as sp
from psiop import PseudoDifferentialOperator

# --- Grid & Test Function Setup ---
N = 256; L = 5.0
x_grid = np.linspace(-L, L, N, endpoint=False)
dx = x_grid[1] - x_grid[0]
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
u = np.exp(-x_grid**2 / 2).astype(complex)

x, xi = sp.symbols('x xi', real=True)

# A genuinely mixed joint symbol:
# 1. sin(x*xi)       -> NUFFT
# 2. 1/(1+(x-xi)^2)  -> AAA
# 3. exp(-(x-xi)^2)  -> Lowrank
p_mixed = sp.sin(x * xi) + 1 / (1 + (x - xi)**2) + sp.exp(-((x - xi)**2) / 8)
op_mixed = PseudoDifferentialOperator(p_mixed, [x], mode='symbol')

print("=" * 70)
print("MONOLITHIC APPROACH: joint_backend='auto' on the WHOLE sum")
print("=" * 70)
t0 = time.time()
# The auto-selector sees the sum, gets confused, tries lowrank, fails, 
# and falls back to direct O(N^2) quadrature.
res_mono = op_mixed.apply(
    u, x_grid, kx, boundary_condition='periodic',
    joint_backend='auto', freq_window=None, clamp=np.inf
)
t_mono = time.time() - t0
print(f"Time: {t_mono:.4f}s (Notice the slowdown due to fallback)")

print("\n" + "=" * 70)
print("HYBRID APPROACH: apply_hybrid() splits and routes individually")
print("=" * 70)
t0 = time.time()
# The hybrid method splits the sum, and routes:
# sin(x*xi) -> NUFFT
# 1/(...)   -> AAA
# exp(...)  -> Lowrank
res_hybrid = op_mixed.apply_hybrid(
    u, x_grid, kx, boundary_condition='periodic',
    freq_window=None, clamp=np.inf
)
t_hybrid = time.time() - t0
print(f"Time: {t_hybrid:.4f}s (Massive speedup!)")

# Verify mathematical equivalence
err = np.max(np.abs(res_mono - res_hybrid))
print(f"\nMax |err| between Monolithic and Hybrid: {err:.3e}")
print("✅ Linearity holds: Op(A+B+C) == Op(A) + Op(B) + Op(C)")

MONOLITHIC APPROACH: joint_backend='auto' on the WHOLE sum
Time: 9.0182s (Notice the slowdown due to fallback)

HYBRID APPROACH: apply_hybrid() splits and routes individually


/home/philippe/psipy/src/psiop.py:3381: UserWarning: Peetre joint residual has been ignored. The result is an asymptotic/local+separable approximation.
  warnings.warn(


Time: 8.4268s (Massive speedup!)

Max |err| between Monolithic and Hybrid: 0.000e+00
✅ Linearity holds: Op(A+B+C) == Op(A) + Op(B) + Op(C)


---
## Architecture notes (this revision)

The joint-residual pipeline is now layered:

1. **Symbolic tier** — `peetre_decomposition(..., classify_joint=True)`:
   local/separable/joint split plus optional backend recommendation
   (`deco['joint_backend']`), cached on `(symbol, separable_local, classify_joint)`.

2. **Normalization tier** — `_resolve_nufft_plan` (grid-free NUFFT plan) and
   `_resolve_joint_representation(joint_symbol, backend, bounds, ...)`:
   maps `{'direct','lowrank','nufft','aaa','auto'}` to a typed dict
   (`separable_pairs`, `nufft_plan`, `aaa_callable`, `direct`,
   `nufft_unrepresentable`, `aaa_unfit`).

3. **Execution tier** — `_apply_joint_residual(...)`: applies a representation
   with quality gates (`joint_max_rel_error`), the periodic-only NUFFT check,
   and automatic fallback to exact direct KN quadrature. `apply_peetre`'s
   joint block is now a thin call to it (this also fixes the previously
   nested/never-called dispatch code), and `apply_hybrid` routes each additive
   joint term through the same machinery.

`print_peetre_decomposition` consumes the same representations: lowrank prints
the separable pairs, nufft/aaa print structural summaries with fit quality,
and unfit representations print the raw joint terms. `clear_cache()` now also
clears `_joint_nufft_cache` and `_joint_aaa_cache`.

---
## Parameter summary

| Parameter | Typical value | Meaning |
|---|---:|---|
| `joint_backend` | `'direct'` | Raw joint residual (exact, expensive O(N²)). |
| `joint_backend` | `'nufft'` | NUFFT phase decomposition (O(N log N)); falls back to direct if unrepresentable. |
| `joint_backend` | `'lowrank'` | Chebyshev/SVD separable pairs (smooth). |
| `joint_backend` | `'aaa'` | Vector-valued AAA rational fit (poles). |
| `joint_backend` | `'auto'` | Intelligent dispatcher based on symbolic structure. |
| `joint_bounds` | `{x: (-5,5), xi: (-30,30)}` | Bounded domain; required for `'lowrank'`/`'aaa'` printing; inferred from grids at `apply()` time. |
| `joint_degree` | `6` | Chebyshev degree per variable (`'lowrank'`). |
| `joint_tol` | `1e-5` | SVD tolerance (`'lowrank'`) / AAA fit tolerance (`'aaa'`). |
| `joint_num_samples` | `10000` | Monte Carlo samples for error diagnostics. |
| `joint_seed` | `42` | Random seed. |
| `joint_max_rel_error` | `None` or float | Max acceptable representation error before direct fallback. |
| `separable_local` | `False`/`True` | Controls local-term representation. |
| `classify_joint` | `False`/`True` | NEW: store the auto-selected backend in `peetre_decomposition()`'s result. |
| `apply_joint` | `True` | If `False`, the joint residual is skipped entirely. |
| `freq_window` | `'gaussian'`/`None` | Frequency windowing (`None` for exact math). |
| `clamp` | `1e6`/`np.inf` | Symbol magnitude clipping (`np.inf` to disable). |
| `use_cache` | `True` | Cache decomposition and low-rank/NUFFT/AAA plans. |

### Backend Strategy & Cost Comparison

| Backend | Strategy | Cost / Complexity |
|---|---|---|
| `'direct'` | Full KN quadrature on the joint symbol | O(N²) |
| `'nufft'` | NUFFT phase decomposition (oscillatory) | O(N log N) |
| `'lowrank'` | Chebyshev/SVD separable pairs (smooth) | O(r · N log N) |
| `'aaa'` | Vector-valued AAA rational fit (poles) | O(N log N) via fast callable |

### AAA-compatible joint symbols (1D)

| Symbol | AAA? | Reason |
|---|---|---|
| `1/(1+(x−ξ)²)` | ✅ | Rational, fixed pole structure in ξ. |
| `1/(ξ² + (x−ξ)² + 1)` | ✅ | Resolvent-shaped, algebraic decay. |
| `sin(x·ξ)` | ❌ | Oscillatory phase (use `'nufft'`). |
| `exp(−(x−ξ)²/8)` | ⚠️ | Gaussian (smooth, better suited for `'lowrank'`). |